## ETAPA 3 - LIMPIEZA BÁSICA DEL DATASET 
### Objetivo En este notebook se realizará un laboratorio práctico de limpieza básica 

Sobre un archivo CSV. Primero se creará una copia del dataset original y se ensuciará de forma intencional con valores nulos y filas duplicadas. Luego se identificarán esos problemas y se eliminarán para obtener un dataset limpio. 

Regla de trabajo El archivo original ubicado en Resultados de la búsqueda en "C:\Proyecto-Risk-Score-OECE\data\raw\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO.csv" no será modificado. Todas las pruebas del laboratorio se realizarán sobre copias guardadas en `data/processed/`

In [1]:
# ============================================================
# FASE 3 - BLOQUE 1
# COPIAR EL DATASET DE ADJUDICACIONES Y ENSUCIARLO A PROPÓSITO
# ============================================================

import pandas as pd
from pathlib import Path

# ============================================================
# 1. DEFINIR LAS RUTAS DE TRABAJO
# ============================================================

# Modifica esta ruta según la ubicación del proyecto en tu computadora
carpeta_proyecto = Path(r"C:\Proyecto-Risk-Score-OECE")

ruta_original = (
    carpeta_proyecto
    / "data"
    / "raw"
    / "CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO.csv"
)

ruta_sucio = (
    carpeta_proyecto
    / "data"
    / "processed"
    / "CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_SUCIO.csv"
)

# Crear la carpeta processed si todavía no existe
ruta_sucio.parent.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. CARGAR EL ARCHIVO ORIGINAL
# ============================================================

# El archivo:
# - usa punto y coma como separador
# - está codificado como UTF-8 con BOM
# - contiene columnas con diferentes tipos de datos

df_original = pd.read_csv(
    ruta_original,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

# ============================================================
# 3. CREAR UNA COPIA DE TRABAJO
# ============================================================

# El archivo original no será modificado
df_sucio = df_original.copy()

# ============================================================
# 4. MOSTRAR INFORMACIÓN INICIAL
# ============================================================

filas_iniciales = df_sucio.shape[0]
columnas_iniciales = df_sucio.shape[1]
nulos_iniciales = df_sucio.isnull().sum().sum()
duplicados_iniciales = df_sucio.duplicated().sum()

print("=" * 65)
print("INFORMACIÓN INICIAL DEL DATASET")
print("=" * 65)

print("Cantidad de filas:", filas_iniciales)
print("Cantidad de columnas:", columnas_iniciales)
print("Dimensión original:", df_sucio.shape)
print("Cantidad inicial de nulos:", nulos_iniciales)
print("Cantidad inicial de duplicados:", duplicados_iniciales)

print("\nColumnas disponibles:")
print(df_sucio.columns.tolist())

# ============================================================
# 5. INSERTAR 20 VALORES NULOS
# ============================================================

# Se utilizarán 20 columnas diferentes de la base.
# Para cada columna se busca una fila que actualmente tenga un dato,
# garantizando así que se añada un nulo nuevo.

columnas_para_nulos = [
    "anio",
    "codigoentidad",
    "entidad_ruc",
    "entidad",
    "tipoentidad",
    "entidad_departamento",
    "codigoconvocatoria",
    "objetocontractual",
    "tipoprocesoseleccion",
    "proceso",
    "descripcion_proceso",
    "n_item",
    "descripcion_item",
    "estado_item",
    "cantidad_adjudicado_item",
    "monto_referencial_item_soles",
    "monto_adjudicado_item_soles",
    "departamento_item",
    "moneda",
    "proveedor"
]

# Guardaremos una relación de las celdas modificadas
celdas_modificadas = []

for numero, columna in enumerate(columnas_para_nulos, start=1):

    # Buscar filas que tengan un valor no nulo en la columna
    filas_con_dato = df_sucio.index[df_sucio[columna].notna()]

    if len(filas_con_dato) == 0:
        print(f"Advertencia: la columna '{columna}' no tiene datos disponibles.")
        continue

    # Elegir una fila diferente para cada modificación
    posicion = (numero - 1) % len(filas_con_dato)
    indice_fila = filas_con_dato[posicion]

    # Guardar el valor original para mostrar evidencia
    valor_original = df_sucio.loc[indice_fila, columna]

    # Insertar el valor nulo
    df_sucio.loc[indice_fila, columna] = None

    celdas_modificadas.append({
        "fila": indice_fila,
        "columna": columna,
        "valor_original": valor_original,
        "valor_nuevo": None
    })

print("\n" + "=" * 65)
print("20 VALORES NULOS INSERTADOS")
print("=" * 65)

for cambio in celdas_modificadas:
    print(
        f"Fila {cambio['fila']} | "
        f"Columna: {cambio['columna']} | "
        f"Valor anterior: {cambio['valor_original']} | "
        f"Valor nuevo: NULL"
    )

# ============================================================
# 6. AGREGAR 20 FILAS DUPLICADAS
# ============================================================

# Seleccionamos las filas 20 a 39 del dataset.
# Se hace una copia para asegurar que sean duplicados exactos.

filas_a_duplicar = df_sucio.iloc[20:40].copy()

# Agregar las 20 filas al final del dataset
df_sucio = pd.concat(
    [df_sucio, filas_a_duplicar],
    ignore_index=True
)

print("\n" + "=" * 65)
print("20 FILAS DUPLICADAS AGREGADAS")
print("=" * 65)

print("Índices originales duplicados:", list(range(20, 40)))
print("Cantidad de filas agregadas:", len(filas_a_duplicar))

# ============================================================
# 7. CALCULAR LOS RESULTADOS DEL ENSUCIAMIENTO
# ============================================================

filas_finales = df_sucio.shape[0]
nulos_finales = df_sucio.isnull().sum().sum()
duplicados_finales = df_sucio.duplicated().sum()

nulos_agregados = nulos_finales - nulos_iniciales
filas_agregadas = filas_finales - filas_iniciales
duplicados_agregados = duplicados_finales - duplicados_iniciales

print("\n" + "=" * 65)
print("RESULTADOS DESPUÉS DE ENSUCIAR EL DATASET")
print("=" * 65)

print("Dimensión original:", df_original.shape)
print("Nueva dimensión:", df_sucio.shape)

print("\nCantidad inicial de nulos:", nulos_iniciales)
print("Cantidad final de nulos:", nulos_finales)
print("Nulos agregados:", nulos_agregados)

print("\nDuplicados iniciales:", duplicados_iniciales)
print("Duplicados finales:", duplicados_finales)
print("Duplicados agregados:", duplicados_agregados)

print("\nFilas agregadas:", filas_agregadas)

# ============================================================
# 8. MOSTRAR LOS NULOS POR COLUMNA
# ============================================================

nulos_por_columna = df_sucio.isnull().sum()

print("\n" + "=" * 65)
print("VALORES NULOS POR COLUMNA")
print("=" * 65)

print(nulos_por_columna)

# Mostrar solamente columnas que tienen nulos
print("\nColumnas que presentan valores nulos:")

print(
    nulos_por_columna[
        nulos_por_columna > 0
    ].sort_values(ascending=False)
)

# ============================================================
# 9. MOSTRAR LAS 20 FILAS DUPLICADAS AGREGADAS
# ============================================================

print("\n" + "=" * 65)
print("MUESTRA DE LAS FILAS AGREGADAS AL FINAL")
print("=" * 65)

display(df_sucio.tail(20))

# ============================================================
# 10. GUARDAR EL DATASET SUCIO
# ============================================================

df_sucio.to_csv(
    ruta_sucio,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 65)
print("ARCHIVO GENERADO CORRECTAMENTE")
print("=" * 65)

print("Archivo original:")
print(ruta_original)

print("\nArchivo sucio:")
print(ruta_sucio)

# ============================================================
# 11. VALIDACIONES FINALES
# ============================================================

assert filas_agregadas == 20, (
    f"Se esperaban 20 filas agregadas, pero se agregaron {filas_agregadas}."
)

assert nulos_agregados == 20, (
    f"Se esperaban 20 nulos nuevos, pero se agregaron {nulos_agregados}."
)

assert duplicados_agregados == 20, (
    f"Se esperaban 20 duplicados nuevos, pero se detectaron "
    f"{duplicados_agregados}."
)

print("\nValidación superada:")
print("✓ Se agregaron exactamente 20 valores nulos.")
print("✓ Se agregaron exactamente 20 filas.")
print("✓ Se generaron exactamente 20 duplicados nuevos.")
print("✓ El archivo original permanece sin modificaciones.")

INFORMACIÓN INICIAL DEL DATASET
Cantidad de filas: 100089
Cantidad de columnas: 26
Dimensión original: (100089, 26)
Cantidad inicial de nulos: 3253
Cantidad inicial de duplicados: 0

Columnas disponibles:
['anio', 'codigoentidad', 'entidad_ruc', 'entidad', 'tipoentidad', 'entidad_departamento', 'codigoconvocatoria', 'objetocontractual', 'tipoprocesoseleccion', 'proceso', 'descripcion_proceso', 'n_item', 'descripcion_item', 'estado_item', 'cantidad_adjudicado_item', 'monto_referencial_item_soles', 'monto_adjudicado_item_soles', 'departamento_item', 'moneda', 'unidad_medida', 'ruc_proveedor', 'proveedor', 'tipo_proveedor', 'fecha_convocatoria', 'fecha_buenapro', 'fecha_consentimiento_bp']

20 VALORES NULOS INSERTADOS
Fila 0 | Columna: anio | Valor anterior: 2018 | Valor nuevo: NULL
Fila 1 | Columna: codigoentidad | Valor anterior: 49.0 | Valor nuevo: NULL
Fila 2 | Columna: entidad_ruc | Valor anterior: 20408454299.0 | Valor nuevo: NULL
Fila 3 | Columna: entidad | Valor anterior: UNIVERSI

,anio,codigoentidad,entidad_ruc,entidad,tipoentidad,entidad_departamento,codigoconvocatoria,objetocontractual,tipoprocesoseleccion,proceso,...,monto_adjudicado_item_soles,departamento_item,moneda,unidad_medida,ruc_proveedor,proveedor,tipo_proveedor,fecha_convocatoria,fecha_buenapro,fecha_consentimiento_bp
100089,2018,24.0,2.013138e+10,MINISTERIO DE TRANSPORTES Y COMUNICACIONES,GOBIERNO NACIONAL,LIMA,501892,Servicio,Contratación Directa,DIRECTA-PROC-24-2018-MTC/10-1,...,"40,000.00",LIMA,Soles,Servicio,20100072751,EMPRESA PERUANA DE SERVICIOS EDITORIALES S.A. ...,Persona Juridica,13/12/2018,13/12/2018,14/12/2018
100090,2018,2409.0,2.010019e+10,SOCIEDAD ELECTRICA DEL SUR OESTE S.A.,FONAFE,AREQUIPA,451224,Bien,Adjudicación Simplificada,AS-SM-26-2018-SEAL-1,...,"217,269.29",AREQUIPA,Soles,Unidad,20298145899,I & T ELECTRIC S.A.C,Persona Juridica,07/06/2018,26/06/2018,04/07/2018
100091,2018,1232.0,2.022696e+10,MUNICIPALIDAD PROVINCIAL DE PATAZ - TAYABAMBA,GOBIERNO LOCAL,LA LIBERTAD,478399,Obra,Adjudicación Simplificada,AS-SM-15-2018-M.P.PATAZ-TAYABAMBA-1,...,"257,914.35",LA LIBERTAD,Soles,Servicio,20477430369,CONSTRUCTORES GENERALES RRK S.A.C.,Persona Juridica,02/10/2018,17/10/2018,18/10/2018
100092,2018,1272.0,2.020464e+10,MUNICIPALIDAD DISTRITAL DE CHAO,GOBIERNO LOCAL,LA LIBERTAD,452109,Consultoría de Obra,Adjudicación Simplificada,AS-SM-3-2018-MDCH/CONSULTORI-1,...,"398,286.68",LA LIBERTAD,Soles,Servicio,20481475342,R&C INGENIEROS CONSULTORES Y CONSTRUCTORES SRL,Persona Juridica,12/06/2018,03/07/2018,12/07/2018
100093,2018,10249.0,2.014436e+10,FUERZA AEREA DEL PERU,GOBIERNO NACIONAL,LIMA,469845,Bien,Regímen Especial,RES-PROC-21-2018-CE2-FAP/SEBAT-1,...,33.79,LIMA,Dólar Norteamericano,Unidad,L0000002900,"SAM EXPORT, CORP",Persona No Domiciliada,24/08/2018,21/11/2018,22/11/2018
100094,2018,1982.0,2.013590e+10,INSTITUTO PERUANO DEL DEPORTE,GOBIERNO NACIONAL,LIMA,453689,Bien,Subasta Inversa Electrónica,SIE-SIE-2-2018-IPD/UL-1,...,"39,600.00",LIMA,Soles,Galon,20518767764,NEGOCIACIONES DANA E.I.R.L.,Persona Juridica,19/06/2018,28/06/2018,09/07/2018
100095,2018,203786.0,2.060222e+10,DIRECCION DE REDES INTEGRADAS DE SALUD LIMA NORTE,GOBIERNO NACIONAL,LIMA,494222,Servicio,Contratación Directa,DIRECTA-PROC-16-2018-DIRIS LN-1,...,"113,000.00",LIMA,Soles,Servicio,20600310870,EDIFIM SOCIEDAD ANONIMA CERRADA - EDIFIM S.A.C.,Persona Juridica,19/11/2018,20/11/2018,21/11/2018
100096,2018,12632.0,2.038196e+10,POLICÍA NACIONAL DEL PERÚ - DIRECCIÓN EJECUTIV...,GOBIERNO NACIONAL,LIMA,439132,Bien,Adjudicación Simplificada,AS-SM-2-2018-ENFPP-PNP-1,...,"47,800.00",LIMA,Soles,Unidad,20102306598,VISTONY COMPAÑIA INDUSTRIAL DEL PERU SOCIEDAD ...,Persona Juridica,12/04/2018,27/04/2018,08/05/2018
100097,2018,2525.0,2.010003e+10,BANCO DE LA NACION,FONAFE,LIMA,431211,Servicio,Contratación Directa,DIRECTA-PROC-1-2018-BN-1,...,"391,983.36",LIMA,Soles,Servicio,20143229816,EMPRESA EDITORA EL COMERCIO S.A.,Persona Juridica,01/03/2018,05/03/2018,06/03/2018
100098,2018,48.0,2.048400e+10,GOBIERNO REGIONAL DE PIURA SEDE CENTRAL,GOBIERNO REGIONAL,PIURA,479893,Bien,Comparación de Precios,COMPRE-SM-2-2018-GRP-ORA-OEC-1,...,"45,700.00",PIURA,Soles,Unidad,10026588702,CALLE RUIZ HECTOR WILFREDO,Persona Natural,21/09/2018,28/09/2018,09/10/2018



ARCHIVO GENERADO CORRECTAMENTE
Archivo original:
C:\Proyecto-Risk-Score-OECE\data\raw\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO.csv

Archivo sucio:
C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_SUCIO.csv

Validación superada:
✓ Se agregaron exactamente 20 valores nulos.
✓ Se agregaron exactamente 20 filas.
✓ Se generaron exactamente 20 duplicados nuevos.
✓ El archivo original permanece sin modificaciones.


In [2]:
# ==========================================
# FASE 3 - BLOQUE 2
# IDENTIFICAR Y LIMPIAR NULOS Y DUPLICADOS
# ==========================================

import pandas as pd

# 1. Definir rutas
ruta_sucio = r"C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_SUCIO.csv"
ruta_limpio = r"C:\Proyecto-Risk-Score-OECE\data\processed\CONOSCE_ADJUDICACIONES_2018_2026_CONSOLIDADO_LIMPIO.csv"

# 2. Cargar el archivo sucio
df_limpieza = pd.read_csv(ruta_sucio, sep=";")

# 3. Verificar el estado antes de limpiar
print("Dimensión del dataset sucio:", df_limpieza.shape)
print("\nCantidad total de nulos antes de limpiar:")
print(df_limpieza.isnull().sum().sum())

print("\nCantidad de nulos por columna antes de limpiar:")
print(df_limpieza.isnull().sum())

print("\nCantidad de filas duplicadas antes de limpiar:")
print(df_limpieza.duplicated().sum())

# 4. Eliminar filas con valores nulos
df_sin_nulos = df_limpieza.dropna()

# 5. Eliminar filas duplicadas
df_limpio = df_sin_nulos.drop_duplicates()

# 6. Verificar el estado después de limpiar
print("\nDESPUÉS DE LA LIMPIEZA")
print("Dimensión del dataset limpio:", df_limpio.shape)

print("\nCantidad total de nulos después de limpiar:")
print(df_limpio.isnull().sum().sum())

print("\nCantidad de filas duplicadas después de limpiar:")
print(df_limpio.duplicated().sum())

# 7. Guardar el archivo limpio
df_limpio.to_csv(ruta_limpio, sep=";", index=False)

print("\nArchivo limpio guardado en:")
print(ruta_limpio)

C:\Users\Iriarte 06\AppData\Local\Temp\ipykernel_36620\170932510.py:13: DtypeWarning: Columns (0,6,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_limpieza = pd.read_csv(ruta_sucio, sep=";")


Dimensión del dataset sucio: (100109, 26)

Cantidad total de nulos antes de limpiar:
3273

Cantidad de nulos por columna antes de limpiar:
anio                               1
codigoentidad                     20
entidad_ruc                       20
entidad                           20
tipoentidad                       20
entidad_departamento              20
codigoconvocatoria                20
objetocontractual                 20
tipoprocesoseleccion              20
proceso                           20
descripcion_proceso               20
n_item                            36
descripcion_item                  41
estado_item                       64
cantidad_adjudicado_item          67
monto_referencial_item_soles      71
monto_adjudicado_item_soles       73
departamento_item               1488
moneda                            75
unidad_medida                     74
ruc_proveedor                     75
proveedor                         77
tipo_proveedor                    73
fecha_conv